# Deep Dive — Data Augmentation

**Module 4: CNNs, Pretrained Models & Deep Dream | Optional Deep Dive**

---

Your CNN learns from the training images it sees. If you only have 50,000 images, you can artificially expand that dataset by applying random transformations: flips, rotations, crops, color changes. The image looks different to the model each time, but the label stays the same.

This is **data augmentation** — one of the most effective and cheapest ways to improve a CNN.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load a raw CIFAR-10 image to show augmentations
raw_data = datasets.CIFAR10('data', train=True, download=True)
sample_img = raw_data[0][0]  # PIL image

plt.figure(figsize=(3, 3))
plt.imshow(sample_img)
plt.title('Original', fontweight='bold')
plt.axis('off')
plt.show()

---

## Common Augmentations

Let's see what each transform does.

In [ ]:
# Show each augmentation
augmentations = {
    'Horizontal Flip': transforms.RandomHorizontalFlip(p=1.0),
    'Random Crop': transforms.RandomCrop(32, padding=4),
    'Rotation (15°)': transforms.RandomRotation(15),
    'Color Jitter': transforms.ColorJitter(brightness=0.3, contrast=0.3,
                                            saturation=0.3, hue=0.1),
    'Random Erasing': transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomErasing(p=1.0, scale=(0.02, 0.2)),
        transforms.ToPILImage(),
    ]),
    'Grayscale': transforms.RandomGrayscale(p=1.0),
}

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

# Original in top-left
axes[0, 0].imshow(sample_img)
axes[0, 0].set_title('Original', fontweight='bold')
axes[0, 0].axis('off')

for ax, (name, aug) in zip(list(axes.flat)[1:7], augmentations.items()):
    augmented = aug(sample_img)
    ax.imshow(augmented)
    ax.set_title(name, fontweight='bold', fontsize=10)
    ax.axis('off')

# Combined
combined = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
])
axes[1, 3].imshow(combined(sample_img))
axes[1, 3].set_title('Combined', fontweight='bold', fontsize=10)
axes[1, 3].axis('off')

plt.suptitle('Data Augmentation Transforms', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Show the same image augmented 8 times
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax in axes.flat:
    ax.imshow(combined(sample_img))
    ax.axis('off')
plt.suptitle('Same Image, 8 Random Augmentations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("Each time the model sees this image, it looks slightly different.")
print("The label doesn't change — it's still the same class.")

---

## Experiment: With vs Without Augmentation

Let's train two identical CNNs on CIFAR-10 — one with augmentation, one without — and compare.

In [ ]:
# Two transform pipelines
transform_basic = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

transform_augmented = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

# Test set: no augmentation (always)
test_data = datasets.CIFAR10('data', train=False, transform=transform_basic)
test_loader = DataLoader(test_data, batch_size=256)

# Two training sets
train_basic = datasets.CIFAR10('data', train=True, transform=transform_basic)
train_augmented = datasets.CIFAR10('data', train=True, transform=transform_augmented)

loader_basic = DataLoader(train_basic, batch_size=64, shuffle=True)
loader_augmented = DataLoader(train_augmented, batch_size=64, shuffle=True)

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def train_and_evaluate(model, train_loader, test_loader, device, epochs=20):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()
    train_losses, test_accs = [], []

    for epoch in range(epochs):
        model.train()
        total, n = 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss_fn(model(images), labels).backward()
            optimizer.step()
            total += loss_fn(model(images), labels).item()
            n += 1
        train_losses.append(total / n)

        model.eval()
        correct, count = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                correct += (model(images).argmax(1) == labels).sum().item()
                count += labels.size(0)
        test_accs.append(correct / count)

        if (epoch + 1) % 5 == 0:
            print(f"  Epoch {epoch+1:2d}  acc={test_accs[-1]:.2%}")

    return train_losses, test_accs


print("Training WITHOUT augmentation:")
torch.manual_seed(42)
model_basic = CNN()
loss_basic, acc_basic = train_and_evaluate(model_basic, loader_basic, test_loader, device)

print("\nTraining WITH augmentation:")
torch.manual_seed(42)
model_aug = CNN()
loss_aug, acc_aug = train_and_evaluate(model_aug, loader_augmented, test_loader, device)

In [ ]:
# Compare
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
epochs = range(1, len(loss_basic) + 1)

axes[0].plot(epochs, loss_basic, 'o-', label='No augmentation', color='#D9534F', linewidth=2, markersize=3)
axes[0].plot(epochs, loss_aug, 's-', label='With augmentation', color='#5CB85C', linewidth=2, markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Training Loss')
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, [a*100 for a in acc_basic], 'o-', label=f'No aug ({acc_basic[-1]:.1%})',
            color='#D9534F', linewidth=2, markersize=3)
axes[1].plot(epochs, [a*100 for a in acc_aug], 's-', label=f'With aug ({acc_aug[-1]:.1%})',
            color='#5CB85C', linewidth=2, markersize=3)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Test Accuracy', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Data Augmentation: With vs Without', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

Notice: with augmentation, training loss is higher (harder to memorize augmented data) but test accuracy is better (better generalization). This is the classic sign that augmentation is working — it's preventing overfitting.

---

## Exercises

1. Try adding `transforms.RandomErasing()` to the augmentation pipeline. Does it help?
2. What augmentations would NOT make sense for CIFAR-10? (Hint: would a vertical flip of a car still look like a car?)
3. Train for 30 epochs instead of 20. Does the gap between augmented and non-augmented get bigger or smaller?

In [ ]:
# Exercises: Your code here
